In [ ]:
# =============================================================================
# STOCHASTIC ATTRACTION, TRANSIENT, AND VALUE-STATE DIAGNOSTICS
# =============================================================================
# Companion analysis for Centipede_Basins_Attraction_v2.ipynb.
#
# This notebook reads one immutable v3 run directory. It treats the binary
# terminal classification as a consistency check and foregrounds continuous
# terminal shares, first-attraction times, exits and re-entries, coarse
# windowed movement, and final policy states. The value-clock section reports
# finite-endpoint diagnostics only; it does not test an asymptotic event.
# =============================================================================


In [ ]:
# =============================================================================
# CELL 1: IMPORTS AND IMMUTABLE RUN SELECTION
# =============================================================================
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib.colors import (
    LinearSegmentedColormap, LogNorm, Normalize, SymLogNorm, TwoSlopeNorm,
)

RUN_ROOT = Path('basin_runs')
SELECTED_RUN_ID = os.environ.get('CENTIPEDE_BASIN_RUN_ID', '').strip()
available_runs = sorted(
    path.name for path in RUN_ROOT.iterdir()
    if path.is_dir() and (path / 'manifest.json').exists()
) if RUN_ROOT.exists() else []

if not SELECTED_RUN_ID and len(available_runs) == 1:
    SELECTED_RUN_ID = available_runs[0]
    print(f'Using the sole available immutable run: {SELECTED_RUN_ID}')
elif not SELECTED_RUN_ID:
    print('Available run IDs:')
    for run_id in available_runs:
        print('  ', run_id)
    raise RuntimeError(
        'Set CENTIPEDE_BASIN_RUN_ID to the exact run directory to diagnose. '
        'Automatic selection is allowed only when exactly one immutable run is available.'
    )

RUN_DIR = RUN_ROOT / SELECTED_RUN_ID
if not RUN_DIR.is_dir():
    raise FileNotFoundError(f'Selected run directory does not exist: {RUN_DIR.resolve()}')

DIAGNOSTIC_TIMESTAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
DIAGNOSTIC_DIR = RUN_DIR / 'diagnostics' / DIAGNOSTIC_TIMESTAMP
DIAGNOSTIC_DIR.mkdir(parents=True, exist_ok=False)
PAPER_ASSET_DIR = Path.cwd().resolve().parent / 'Drafts'
PAPER_ASSET_DIR.mkdir(parents=True, exist_ok=True)
print(f'Selected run: {RUN_DIR.resolve()}')
print(f'Diagnostic outputs: {DIAGNOSTIC_DIR.resolve()}')
print(f'Stable manuscript assets: {PAPER_ASSET_DIR.resolve()}')


In [ ]:
# =============================================================================
# CELL 2: LOAD THE V3 RUN PACKAGE
# =============================================================================
manifest = json.loads((RUN_DIR / 'manifest.json').read_text(encoding='utf-8'))
if manifest.get('schema_version') != 'centipede.stochastic-attraction.v3':
    raise ValueError(f"Unsupported schema: {manifest.get('schema_version')}")
if manifest.get('status') != 'complete':
    raise ValueError(f"Run is not complete: status={manifest.get('status')}")

grid_path = RUN_DIR / 'aggregate' / 'grid_summary.csv'
replicate_path = RUN_DIR / 'aggregate' / 'replicate_summary.csv'
if not grid_path.is_file() or not replicate_path.is_file():
    raise FileNotFoundError('The selected run lacks its aggregate CSV files')

df_grid = pd.read_csv(grid_path)
df_replicates = pd.read_csv(replicate_path)

POLICY_ORDER = ['epsilon_greedy', 'thompson', 'exp3']
POLICY_LABELS = {
    'epsilon_greedy': 'Epsilon-Greedy',
    'thompson': 'Thompson Sampling',
    'exp3': 'Repaired Anytime Exp3',
}
PARAMETER_LABELS = {
    'epsilon_greedy': 'early-to-late Q-bias (beta)',
    'thompson': 'early-to-late prior-mean bias (mu)',
    'exp3': 'stage-0-to-stage-2 weight shift (alpha)',
}
OUTCOME_LABELS = {0: '(0,0)', 1: '(1,1)', 2: '(2,2)', 3: 'Mixed/other'}
PROFILE_COLORS = {
    '(0,0)': '#2166ac', '(1,1)': '#e08214', '(2,2)': '#1b9e77',
    'Off diagonal': '#7b3294', 'Residual': '#b2182b',
}

PAYOFF_P1 = np.asarray(manifest['game']['payoff_matrix_p1'], dtype=np.float64)
PAYOFF_P2 = np.asarray(manifest['game']['payoff_matrix_p2'], dtype=np.float64)
if PAYOFF_P1.shape != (3, 3) or PAYOFF_P2.shape != (3, 3):
    raise ValueError('These diagnostics require the saved three-stage game')

print(f"Run status: {manifest['status']}; schema: {manifest['schema_version']}")
print(f"Grid rows: {len(df_grid):,}; replicate rows: {len(df_replicates):,}")


In [ ]:
# =============================================================================
# CELL 3: DATA-INTEGRITY CHECKS
# =============================================================================
def validate_run(df_grid, df_replicates, manifest):
    findings = []
    expected_cells = int(manifest['grid_resolution']) ** 2
    allowed_horizons = set(manifest['horizon_schedule'])
    for policy in POLICY_ORDER:
        grid = df_grid[df_grid['policy'] == policy]
        reps = df_replicates[df_replicates['policy'] == policy]
        probabilities = grid[['p_stage0', 'p_stage1', 'p_stage2', 'p_mixed']].to_numpy()
        grouped_counts = reps.groupby(['grid_i', 'grid_j']).size()
        expected_counts = grid.set_index(['grid_i', 'grid_j'])['n_replicates'].astype(int)
        cell_files = list((RUN_DIR / 'cells' / policy).glob('cell_*.npz'))
        findings.append({
            'policy': POLICY_LABELS[policy],
            'grid_cells': len(grid),
            'expected_cells': expected_cells,
            'cell_files_complete': len(cell_files) == expected_cells,
            'probability_sums_valid': bool(np.allclose(probabilities.sum(axis=1), 1.0)),
            'probabilities_in_unit_interval': bool(np.all((probabilities >= 0) & (probabilities <= 1))),
            'replicate_rows_match': bool(grouped_counts.equals(expected_counts)),
            'horizons_valid': bool(set(grid['final_horizon'].astype(int)).issubset(allowed_horizons)),
            'stable_fraction_valid': bool(grid['stable_fraction'].between(0, 1).all()),
        })
    return pd.DataFrame(findings)


df_validation = validate_run(df_grid, df_replicates, manifest)
display(df_validation)
boolean_columns = [
    column for column in df_validation
    if column not in ('policy', 'grid_cells', 'expected_cells')
]
if not df_validation[boolean_columns].all().all():
    raise ValueError('Run package failed one or more integrity checks')


In [ ]:
# =============================================================================
# CELL 4: CONTINUOUS TERMINAL AND ATTRACTION SUMMARIES
# =============================================================================
df_replicates = df_replicates.copy()
df_replicates['reached_attraction_region'] = df_replicates['first_dominance_time'] >= 0
df_replicates['first_attraction_time'] = df_replicates['first_dominance_time'].where(
    df_replicates['first_dominance_time'] >= 0, np.nan
)
df_replicates['exited_after_entry'] = df_replicates['dominance_exits'] > 0
df_replicates['reentries'] = np.maximum(df_replicates['dominance_entries'] - 1, 0)
df_replicates['last_exit_time'] = df_replicates['last_dominance_exit'].where(
    df_replicates['last_dominance_exit'] >= 0, np.nan
)
df_replicates['terminal_residual_share'] = 1.0 - df_replicates['terminal_stage0_share']

def q10(series):
    return series.quantile(0.10)

def q90(series):
    return series.quantile(0.90)

cell_dynamics = (
    df_replicates.groupby(['policy', 'grid_i', 'grid_j'], as_index=False)
    .agg(
        mean_terminal_stage0_share=('terminal_stage0_share', 'mean'),
        median_terminal_stage0_share=('terminal_stage0_share', 'median'),
        p10_terminal_stage0_share=('terminal_stage0_share', q10),
        mean_terminal_off_diagonal_share=('terminal_off_diagonal_share', 'mean'),
        mean_terminal_residual_share=('terminal_residual_share', 'mean'),
        fraction_reaching_attraction=('reached_attraction_region', 'mean'),
        median_first_attraction_time=('first_attraction_time', 'median'),
        p10_first_attraction_time=('first_attraction_time', q10),
        p90_first_attraction_time=('first_attraction_time', q90),
        fraction_exiting_after_entry=('exited_after_entry', 'mean'),
        mean_reentries=('reentries', 'mean'),
        mean_dominance_exits=('dominance_exits', 'mean'),
        median_last_exit_time=('last_exit_time', 'median'),
    )
)
df_grid = df_grid.merge(
    cell_dynamics, on=['policy', 'grid_i', 'grid_j'], how='left', validate='one_to_one'
)
if df_grid['mean_terminal_stage0_share'].isna().any():
    raise ValueError('Continuous cell summaries failed to cover the complete grid')

df_grid['first_attraction_spread'] = (
    df_grid['p90_first_attraction_time'] - df_grid['p10_first_attraction_time']
)
print('Continuous terminal and attraction summaries added to the grid.')


In [ ]:
# =============================================================================
# CELL 5: GRID-PLOTTING HELPERS
# =============================================================================
def grid_matrix(policy_frame, value_column):
    resolution = int(manifest['grid_resolution'])
    matrix = np.full((resolution, resolution), np.nan)
    for row in policy_frame.itertuples(index=False):
        matrix[int(row.grid_i), int(row.grid_j)] = getattr(row, value_column)
    return matrix


def parameter_extent(policy_frame):
    p1 = np.sort(policy_frame['bias_p1'].unique())
    p2 = np.sort(policy_frame['bias_p2'].unique())
    return (p1[0], p1[-1], p2[0], p2[-1])


def plot_three_policy_grids(value_column, title, colorbar_label, filename,
                            cmap='viridis', vmin=None, vmax=None, norm=None,
                            manuscript_filename=None):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), constrained_layout=True)
    image = None
    for ax, policy in zip(axes, POLICY_ORDER):
        frame = df_grid[df_grid['policy'] == policy]
        image_kwargs = {'cmap': cmap, 'aspect': 'auto', 'interpolation': 'nearest'}
        if norm is None:
            image_kwargs.update(vmin=vmin, vmax=vmax)
        else:
            image_kwargs['norm'] = norm
        image = ax.imshow(
            grid_matrix(frame, value_column).T,
            origin='lower', extent=parameter_extent(frame), **image_kwargs,
        )
        ax.set_title(POLICY_LABELS[policy])
        ax.set_xlabel(f"Player 1 {PARAMETER_LABELS[policy]}")
        ax.set_ylabel(f"Player 2 {PARAMETER_LABELS[policy]}")
    fig.colorbar(image, ax=axes, label=colorbar_label, shrink=0.88)
    fig.suptitle(title, fontsize=15, y=1.03)
    fig.savefig(DIAGNOSTIC_DIR / filename, dpi=300, bbox_inches='tight')
    if manuscript_filename is not None:
        fig.savefig(PAPER_ASSET_DIR / manuscript_filename, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)


def plot_policy_specific_grids(specs, title, filename):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), constrained_layout=True)
    for ax, policy in zip(axes, POLICY_ORDER):
        frame = df_grid[df_grid['policy'] == policy]
        spec = specs[policy]
        image = ax.imshow(
            grid_matrix(frame, spec['column']).T,
            origin='lower', extent=parameter_extent(frame), aspect='auto',
            interpolation='nearest', cmap=spec.get('cmap', 'viridis'),
            norm=spec.get('norm'),
        )
        ax.set_title(f"{POLICY_LABELS[policy]}\n{spec['subtitle']}")
        ax.set_xlabel(f"Player 1 {PARAMETER_LABELS[policy]}")
        ax.set_ylabel(f"Player 2 {PARAMETER_LABELS[policy]}")
        fig.colorbar(image, ax=ax, label=spec['colorbar'], shrink=0.82)
    fig.suptitle(title, fontsize=15, y=1.03)
    fig.savefig(DIAGNOSTIC_DIR / filename, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)


def positive_lognorm(series):
    values = np.asarray(series, dtype=np.float64)
    positive = values[np.isfinite(values) & (values > 0)]
    if len(positive) == 0:
        return LogNorm(vmin=1e-9, vmax=1.0), 1e-9
    floor = float(positive.min())
    ceiling = float(positive.max())
    if np.isclose(floor, ceiling):
        floor = max(floor / 10.0, 1e-12)
        ceiling *= 1.1
    return LogNorm(vmin=floor, vmax=ceiling), floor


def truncated_cmap(name, low=0.10, high=0.88):
    base = plt.get_cmap(name)
    colors = base(np.linspace(low, high, 256))
    return LinearSegmentedColormap.from_list(
        f'{name}_print_{low:.2f}_{high:.2f}', colors
    )


PRINT_CMAPS = {
    'timing': truncated_cmap('YlGnBu', 0.12, 0.84),
    'spread': truncated_cmap('YlOrRd', 0.10, 0.84),
    'attraction': truncated_cmap('YlGn', 0.12, 0.82),
    'residual': truncated_cmap('YlOrRd', 0.10, 0.86),
    'off_diagonal': truncated_cmap('PuBu', 0.12, 0.84),
    'exit': truncated_cmap('YlOrRd', 0.10, 0.84),
    'reentry': truncated_cmap('PuRd', 0.10, 0.82),
    'state': truncated_cmap('YlGnBu', 0.12, 0.84),
    'positive': truncated_cmap('YlGnBu', 0.12, 0.84),
    'nonnegative': truncated_cmap('PuBuGn', 0.12, 0.84),
    'diverging': truncated_cmap('RdBu_r', 0.08, 0.92),
}


print('Grid plotting helpers and print-safe color gradients defined.')


In [ ]:
# =============================================================================
# CELL 6: POLICY-LEVEL SUMMARY
# =============================================================================
def summarize_policies(df_grid, df_replicates):
    rows = []
    for policy in POLICY_ORDER:
        grid = df_grid[df_grid['policy'] == policy]
        reps = df_replicates[df_replicates['policy'] == policy]
        reached = reps['first_attraction_time'].dropna()
        rows.append({
            'Policy': POLICY_LABELS[policy],
            'Grid cells': len(grid),
            'Replicates': len(reps),
            'Mean terminal (0,0) share': reps['terminal_stage0_share'].mean(),
            'Fraction reaching attraction': reps['reached_attraction_region'].mean(),
            'Minimum replicate terminal (0,0) share': reps['terminal_stage0_share'].min(),
            'Mean terminal off-diagonal share': reps['terminal_off_diagonal_share'].mean(),
            'Median first-attraction time': reached.median(),
            '90th percentile first-attraction time': reached.quantile(0.90),
            'Fraction later exiting': reps['exited_after_entry'].mean(),
            'Mean re-entries': reps['reentries'].mean(),
        })
    return pd.DataFrame(rows)


df_policy_summary = summarize_policies(df_grid, df_replicates)
display(df_policy_summary)
print('Terminal classification is reported later as a consistency check; continuous shares lead the analysis.')


In [ ]:
# =============================================================================
# CELL 7: MEDIAN FIRST-ATTRACTION-TIME MAPS
# =============================================================================
attraction_times = df_grid['median_first_attraction_time']
attraction_norm = SymLogNorm(
    linthresh=1.0, vmin=0.0, vmax=float(attraction_times.max()), base=10
)
plot_three_policy_grids(
    'median_first_attraction_time',
    'Median Time to First Sustained Stage-0 Concentration',
    'Round at start of first 500-round window with at least 90% (0,0) play',
    'median_first_attraction_time_panels.png',
    cmap=PRINT_CMAPS['timing'], norm=attraction_norm,
    manuscript_filename='Figure 6.1 median first-attraction time.png',
)
print('A value of zero means the first 500 rounds already satisfy the 90% threshold.')

spread_norm, _ = positive_lognorm(df_grid['first_attraction_spread'])
plot_three_policy_grids(
    'first_attraction_spread',
    'Within-Cell Variation in First-Attraction Time',
    '90th minus 10th percentile first-attraction time',
    'first_attraction_time_spread_panels.png',
    cmap=PRINT_CMAPS['spread'], norm=spread_norm,
    manuscript_filename='Figure 6.2 first-attraction-time spread.png',
)


In [ ]:
# =============================================================================
# CELL 8: CONTINUOUS TERMINAL ATTRACTION AND RESIDUAL EXPLORATION
# =============================================================================
terminal_min = float(df_grid['mean_terminal_stage0_share'].min())
terminal_vmin = max(0.0, terminal_min - max(0.001, 0.05 * (1.0 - terminal_min)))
plot_three_policy_grids(
    'mean_terminal_stage0_share',
    'Continuous Terminal Share of the Backward-Induction Outcome',
    'Mean terminal-quarter share of (0,0) play',
    'continuous_terminal_stage0_share_panels.png',
    cmap=PRINT_CMAPS['attraction'], vmin=terminal_vmin, vmax=1.0,
)

residual_norm, residual_floor = positive_lognorm(df_grid['mean_terminal_residual_share'])
df_grid['terminal_residual_share_for_plot'] = np.maximum(
    df_grid['mean_terminal_residual_share'], residual_floor
)
plot_three_policy_grids(
    'terminal_residual_share_for_plot',
    'Terminal Play Outside the Backward-Induction Outcome',
    'Mean terminal-quarter share outside (0,0), logarithmic scale',
    'terminal_residual_exploration_panels.png',
    cmap=PRINT_CMAPS['residual'], norm=residual_norm,
)

offdiag_norm, offdiag_floor = positive_lognorm(df_grid['mean_terminal_off_diagonal_share'])
df_grid['terminal_off_diagonal_share_for_plot'] = np.maximum(
    df_grid['mean_terminal_off_diagonal_share'], offdiag_floor
)
plot_three_policy_grids(
    'terminal_off_diagonal_share_for_plot',
    'Terminal Off-Diagonal Play',
    'Mean terminal-quarter off-diagonal share, logarithmic scale',
    'terminal_off_diagonal_share_panels.png',
    cmap=PRINT_CMAPS['off_diagonal'], norm=offdiag_norm,
)


In [ ]:
# =============================================================================
# CELL 9: EXIT AND RE-ENTRY MAPS
# =============================================================================
exit_max = max(0.01, float(df_grid['fraction_exiting_after_entry'].max()))
plot_three_policy_grids(
    'fraction_exiting_after_entry',
    'Probability of Leaving Stage-0 Concentration After First Entry',
    'Fraction of replicated trajectories with at least one later exit',
    'post_entry_exit_probability_panels.png',
    cmap=PRINT_CMAPS['exit'], vmin=0.0, vmax=exit_max,
)

reentry_max = max(1.0, float(df_grid['mean_reentries'].max()))
reentry_norm = SymLogNorm(linthresh=0.05, vmin=0.0, vmax=reentry_max, base=10)
plot_three_policy_grids(
    'mean_reentries',
    'Repeated Returns to Stage-0 Concentration',
    'Mean number of re-entries after the first entry',
    'mean_reentry_count_panels.png',
    cmap=PRINT_CMAPS['reentry'], norm=reentry_norm,
)


In [ ]:
# =============================================================================
# CELL 10: LOAD FINAL POLICY STATES AND VALUE-CLOCK COMPARISONS
# =============================================================================
final_cell_rows = []
final_state_samples = {}
value_clock_cell_rows = []
value_clock_samples = {}
lambdas = {
    1: np.array([1.0, 0.0, 0.0]),
    2: np.array([0.25, 0.75, 0.0]),
}

for row in df_grid.itertuples(index=False):
    policy = row.policy
    path = RUN_DIR / 'cells' / policy / f'cell_{int(row.grid_i):02d}_{int(row.grid_j):02d}.npz'
    with np.load(path, allow_pickle=False) as archive:
        location = archive['final_state_location'].astype(np.float64)
        scale = archive['final_state_scale'].astype(np.float64)
        action_counts = archive['final_action_counts'].astype(np.float64)
        total_joint_counts = archive['total_joint_counts'].astype(np.float64)

    stage0_margin = location[:, :, 0] - np.max(location[:, :, 1:], axis=2)
    count_totals = action_counts.sum(axis=2)
    stage0_action_share = np.divide(
        action_counts[:, :, 0], count_totals,
        out=np.zeros_like(count_totals), where=count_totals > 0,
    )
    selection_p0 = scale[:, :, 0] if policy == 'exp3' else np.full_like(stage0_margin, np.nan)
    final_cell_rows.append({
        'policy': policy,
        'grid_i': int(row.grid_i),
        'grid_j': int(row.grid_j),
        'median_stage0_margin': float(np.median(stage0_margin)),
        'p10_stage0_margin': float(np.quantile(stage0_margin, 0.10)),
        'fraction_stage0_ranked_highest': float(np.mean(stage0_margin > 0)),
        'mean_final_stage0_action_share': float(np.mean(stage0_action_share)),
        'median_stage0_selection_probability': (
            float(np.nanmedian(selection_p0)) if policy == 'exp3' else np.nan
        ),
    })

    for player in range(2):
        for stage in range(3):
            key = (policy, player + 1, stage)
            bucket = final_state_samples.setdefault(
                key, {'location': [], 'scale': [], 'count': []}
            )
            bucket['location'].append(location[:, player, stage])
            bucket['scale'].append(scale[:, player, stage])
            bucket['count'].append(action_counts[:, player, stage])

    if policy not in ('epsilon_greedy', 'thompson'):
        continue

    opponent_p2 = total_joint_counts.sum(axis=1)
    opponent_p2 /= opponent_p2.sum(axis=1, keepdims=True)
    opponent_p1 = total_joint_counts.sum(axis=2)
    opponent_p1 /= opponent_p1.sum(axis=1, keepdims=True)
    global_values_p1 = opponent_p2 @ PAYOFF_P1.T
    global_values_p2 = opponent_p1 @ PAYOFF_P2

    for k, lower_mix in lambdas.items():
        estimated_p1 = location[:, 0, :] @ lower_mix - location[:, 0, k]
        estimated_p2 = location[:, 1, :] @ lower_mix - location[:, 1, k]
        global_p1 = global_values_p1 @ lower_mix - global_values_p1[:, k]
        global_p2 = global_values_p2 @ lower_mix - global_values_p2[:, k]
        estimated = np.concatenate([estimated_p1, estimated_p2])
        global_contrast = np.concatenate([global_p1, global_p2])
        discrepancy = estimated - global_contrast
        value_clock_cell_rows.append({
            'policy': policy,
            'grid_i': int(row.grid_i),
            'grid_j': int(row.grid_j),
            'stage_k': k,
            'fraction_estimated_contrast_positive': float(np.mean(estimated > 0)),
            'median_estimated_contrast': float(np.median(estimated)),
            'fraction_discrepancy_nonnegative': float(np.mean(discrepancy >= 0)),
            'median_global_contrast': float(np.median(global_contrast)),
            'median_discrepancy': float(np.median(discrepancy)),
        })
        sample_bucket = value_clock_samples.setdefault(
            (policy, k), {'estimated': [], 'global': [], 'discrepancy': []}
        )
        sample_bucket['estimated'].append(estimated)
        sample_bucket['global'].append(global_contrast)
        sample_bucket['discrepancy'].append(discrepancy)

df_final_state_cells = pd.DataFrame(final_cell_rows)
df_grid = df_grid.merge(
    df_final_state_cells, on=['policy', 'grid_i', 'grid_j'],
    how='left', validate='one_to_one',
)

final_state_summary_rows = []
for (policy, player, stage), samples in final_state_samples.items():
    location = np.concatenate(samples['location'])
    scale = np.concatenate(samples['scale'])
    count = np.concatenate(samples['count'])
    final_state_summary_rows.append({
        'Policy': POLICY_LABELS[policy],
        'Player': player,
        'Stage': stage,
        'Mean final location': location.mean(),
        'Median final location': np.median(location),
        'Mean final scale': scale.mean(),
        'Mean action count': count.mean(),
    })
df_final_state_summary = pd.DataFrame(final_state_summary_rows)
df_value_clock_cells = pd.DataFrame(value_clock_cell_rows)

value_clock_summary_rows = []
for (policy, k), samples in value_clock_samples.items():
    estimated = np.concatenate(samples['estimated'])
    global_contrast = np.concatenate(samples['global'])
    discrepancy = np.concatenate(samples['discrepancy'])
    value_clock_summary_rows.append({
        'Policy': POLICY_LABELS[policy],
        'Stage comparison k': k,
        'Final player states': len(estimated),
        'Fraction estimated contrast positive': np.mean(estimated > 0),
        'Median estimated contrast': np.median(estimated),
        'Median global-clock contrast': np.median(global_contrast),
        'Fraction finite discrepancy nonnegative': np.mean(discrepancy >= 0),
        'Median finite discrepancy': np.median(discrepancy),
    })
df_value_clock_summary = pd.DataFrame(value_clock_summary_rows)
print('Final policy states and finite value-clock comparisons loaded.')


In [ ]:
# =============================================================================
# CELL 11: FINAL VALUE-STATE DIAGNOSTICS
# =============================================================================
def data_norm(series, include_zero=False):
    values = np.asarray(series, dtype=np.float64)
    values = values[np.isfinite(values)]
    low, high = float(values.min()), float(values.max())
    if include_zero:
        low = min(0.0, low)
    if np.isclose(low, high):
        high = low + 1.0
    return Normalize(vmin=low, vmax=high)


final_specs = {
    'epsilon_greedy': {
        'column': 'median_stage0_margin',
        'subtitle': 'Median Q(0) minus max[Q(1), Q(2)]',
        'colorbar': 'Estimated-payoff margin',
        'cmap': PRINT_CMAPS['state'],
        'norm': data_norm(
            df_grid.loc[df_grid.policy == 'epsilon_greedy', 'median_stage0_margin'], True
        ),
    },
    'thompson': {
        'column': 'median_stage0_margin',
        'subtitle': 'Median posterior mean(0) minus later-stage maximum',
        'colorbar': 'Posterior-mean margin',
        'cmap': PRINT_CMAPS['state'],
        'norm': data_norm(
            df_grid.loc[df_grid.policy == 'thompson', 'median_stage0_margin'], True
        ),
    },
    'exp3': {
        'column': 'median_stage0_selection_probability',
        'subtitle': 'Median final selection probability of stage 0',
        'colorbar': 'Selection probability',
        'cmap': PRINT_CMAPS['state'],
        'norm': Normalize(vmin=0.0, vmax=1.0),
    },
}
plot_policy_specific_grids(
    final_specs,
    'Final Policy-State Preference for Stage 0',
    'final_policy_state_stage0_panels.png',
)
display(df_final_state_summary.sort_values(['Policy', 'Player', 'Stage']))


In [ ]:
# =============================================================================
# CELL 12: PROOF-ORIENTED MIXED VALUE-CLOCK DIAGNOSTICS
# =============================================================================
def plot_value_clock_panels(column, title, colorbar_label, filename,
                            cmap='viridis', norm=None, vmin=None, vmax=None):
    policies = ['epsilon_greedy', 'thompson']
    fig, axes = plt.subplots(2, 2, figsize=(13, 11), constrained_layout=True)
    image = None
    for row_index, policy in enumerate(policies):
        grid_frame = df_grid[df_grid['policy'] == policy]
        for column_index, stage_k in enumerate((1, 2)):
            ax = axes[row_index, column_index]
            frame = df_value_clock_cells[
                (df_value_clock_cells['policy'] == policy)
                & (df_value_clock_cells['stage_k'] == stage_k)
            ]
            image_kwargs = {'cmap': cmap, 'aspect': 'auto', 'interpolation': 'nearest'}
            if norm is None:
                image_kwargs.update(vmin=vmin, vmax=vmax)
            else:
                image_kwargs['norm'] = norm
            image = ax.imshow(
                grid_matrix(frame, column).T,
                origin='lower', extent=parameter_extent(grid_frame), **image_kwargs,
            )
            ax.set_title(f"{POLICY_LABELS[policy]}: lower-stage mixture versus stage {stage_k}")
            ax.set_xlabel(f"Player 1 {PARAMETER_LABELS[policy]}")
            ax.set_ylabel(f"Player 2 {PARAMETER_LABELS[policy]}")
    fig.colorbar(image, ax=axes, label=colorbar_label, shrink=0.86)
    fig.suptitle(title, fontsize=15)
    fig.savefig(DIAGNOSTIC_DIR / filename, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)


plot_value_clock_panels(
    'fraction_estimated_contrast_positive',
    'Final Estimated Mixed Dominance Contrasts',
    'Fraction of final player-replicate states with positive estimated contrast',
    'mixed_value_clock_positive_contrast_panels.png',
    cmap=PRINT_CMAPS['positive'], vmin=0.0, vmax=1.0,
)
plot_value_clock_panels(
    'fraction_discrepancy_nonnegative',
    'Finite-Endpoint Mixed Value-Clock Discrepancy',
    'Fraction with estimated minus global-clock contrast at least zero',
    'mixed_value_clock_nonnegative_discrepancy_panels.png',
    cmap=PRINT_CMAPS['nonnegative'], vmin=0.0, vmax=1.0,
)

discrepancy_values = df_value_clock_cells['median_discrepancy'].to_numpy()
discrepancy_limit = float(np.max(np.abs(discrepancy_values[np.isfinite(discrepancy_values)])))
discrepancy_limit = max(discrepancy_limit, 1e-9)
plot_value_clock_panels(
    'median_discrepancy',
    'Median Finite Mixed Value-Clock Discrepancy',
    'Estimated contrast minus global-clock contrast',
    'mixed_value_clock_median_discrepancy_panels.png',
    cmap=PRINT_CMAPS['diverging'], norm=TwoSlopeNorm(
        vmin=-discrepancy_limit, vcenter=0.0, vmax=discrepancy_limit
    ),
)
display(df_value_clock_summary)
print(
    'These are final-horizon proxies. PublicationDmix is an asymptotic liminf event; '
    'the figures neither verify nor refute its occurrence.'
)


In [ ]:
# =============================================================================
# CELL 13: DELIBERATE TEMPORAL-PROFILE CELL SELECTION
# =============================================================================
CATEGORY_ORDER = [
    'Near neutral',
    'Strongly late-biased',
    'Asymmetric',
    'Slowest median attraction',
    'Least terminally concentrated',
]
selection_rows = []

for policy in POLICY_ORDER:
    frame = df_grid[df_grid['policy'] == policy].copy()
    frame['neutral_distance'] = frame['bias_p1'].abs() + frame['bias_p2'].abs()
    neutral = frame.sort_values(
        ['neutral_distance', 'grid_i', 'grid_j'], ascending=[True, True, True]
    ).iloc[0]
    late = frame.sort_values(
        ['bias_p1', 'bias_p2'], ascending=[False, False]
    ).iloc[0]
    asymmetric = frame.sort_values(
        ['bias_p1', 'bias_p2'], ascending=[False, True]
    ).iloc[0]
    slowest = frame.sort_values(
        ['median_first_attraction_time', 'mean_terminal_stage0_share'],
        ascending=[False, True],
    ).iloc[0]
    least_candidates = frame[np.isclose(
        frame['mean_terminal_stage0_share'], frame['mean_terminal_stage0_share'].min()
    )].copy()
    already_selected = {
        (int(selected.grid_i), int(selected.grid_j))
        for selected in (neutral, late, asymmetric, slowest)
    }
    unused_least = least_candidates[
        ~least_candidates.apply(
            lambda candidate: (int(candidate.grid_i), int(candidate.grid_j)) in already_selected,
            axis=1,
        )
    ]
    if len(unused_least):
        least_candidates = unused_least
    least = least_candidates.sort_values(
        ['mean_terminal_stage0_share', 'mean_terminal_off_diagonal_share',
         'median_first_attraction_time'],
        ascending=[True, False, False],
    ).iloc[0]

    for category, selected in zip(
        CATEGORY_ORDER, (neutral, late, asymmetric, slowest, least)
    ):
        selection_rows.append({
            'policy': policy,
            'policy_label': POLICY_LABELS[policy],
            'category': category,
            'grid_i': int(selected.grid_i),
            'grid_j': int(selected.grid_j),
            'bias_p1': float(selected.bias_p1),
            'bias_p2': float(selected.bias_p2),
            'median_first_attraction_time': float(selected.median_first_attraction_time),
            'mean_terminal_stage0_share': float(selected.mean_terminal_stage0_share),
            'mean_terminal_off_diagonal_share': float(selected.mean_terminal_off_diagonal_share),
        })

df_selected_cells = pd.DataFrame(selection_rows)
display(df_selected_cells)
print('Categories may identify the same cell when one initialization is extreme on several diagnostics.')


In [ ]:
# =============================================================================
# CELL 14: WINDOWED TEMPORAL PROFILES FOR SELECTED CELLS
# =============================================================================
temporal_rows = []
window_rounds = int(manifest['diagnostic_window_rounds'])

for selected in df_selected_cells.itertuples(index=False):
    cell_path = (
        RUN_DIR / 'cells' / selected.policy
        / f'cell_{selected.grid_i:02d}_{selected.grid_j:02d}.npz'
    )
    with np.load(cell_path, allow_pickle=False) as archive:
        window_counts = archive['diagnostic_joint_counts'].astype(np.float64)
    pooled = window_counts.sum(axis=0)
    totals = pooled.sum(axis=(1, 2))
    shares = {
        '(0,0)': pooled[:, 0, 0] / totals,
        '(1,1)': pooled[:, 1, 1] / totals,
        '(2,2)': pooled[:, 2, 2] / totals,
    }
    shares['Off diagonal'] = (
        1.0 - shares['(0,0)'] - shares['(1,1)'] - shares['(2,2)']
    )
    shares['Residual'] = 1.0 - shares['(0,0)']
    for window_index in range(len(pooled)):
        for outcome, values in shares.items():
            temporal_rows.append({
                'policy': selected.policy,
                'category': selected.category,
                'grid_i': selected.grid_i,
                'grid_j': selected.grid_j,
                'bias_p1': selected.bias_p1,
                'bias_p2': selected.bias_p2,
                'window_index': window_index + 1,
                'window_end_round': (window_index + 1) * window_rounds,
                'outcome': outcome,
                'share': float(values[window_index]),
            })

df_temporal_profiles = pd.DataFrame(temporal_rows)

fig, axes = plt.subplots(5, 3, figsize=(18, 20), constrained_layout=True, sharey=True)
for row_index, category in enumerate(CATEGORY_ORDER):
    for column_index, policy in enumerate(POLICY_ORDER):
        ax = axes[row_index, column_index]
        frame = df_temporal_profiles[
            (df_temporal_profiles['category'] == category)
            & (df_temporal_profiles['policy'] == policy)
        ]
        for outcome in ('(0,0)', '(1,1)', '(2,2)', 'Off diagonal'):
            series = frame[frame['outcome'] == outcome]
            ax.plot(
                series['window_end_round'], series['share'], marker='o', linewidth=1.8,
                color=PROFILE_COLORS[outcome], label=outcome,
            )
        selected = df_selected_cells[
            (df_selected_cells.category == category)
            & (df_selected_cells.policy == policy)
        ].iloc[0]
        ax.set_title(
            f"{POLICY_LABELS[policy]}: {category}\n"
            f"biases=({selected.bias_p1:.3g}, {selected.bias_p2:.3g})",
            fontsize=10,
        )
        ax.set_ylim(0.0, 1.0)
        ax.grid(alpha=0.35, color='#c7dcef')
        if column_index == 0:
            ax.set_ylabel('Window share')
        if row_index == len(CATEGORY_ORDER) - 1:
            ax.set_xlabel('Window end round')
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(
    handles, labels, loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.012),
    frameon=False,
)
fig.suptitle('Coarse Joint-Action Profiles at Deliberately Selected Cells', fontsize=16, y=1.01)
fig.savefig(DIAGNOSTIC_DIR / 'selected_cell_windowed_joint_profiles.png', dpi=240, bbox_inches='tight')
plt.show()
plt.close(fig)

positive_profile_shares = df_temporal_profiles.loc[
    (df_temporal_profiles['outcome'] != '(0,0)')
    & (df_temporal_profiles['share'] > 0), 'share'
]
profile_floor = float(positive_profile_shares.min()) if len(positive_profile_shares) else 1e-9
fig, axes = plt.subplots(5, 3, figsize=(18, 20), constrained_layout=True, sharey=True)
for row_index, category in enumerate(CATEGORY_ORDER):
    for column_index, policy in enumerate(POLICY_ORDER):
        ax = axes[row_index, column_index]
        frame = df_temporal_profiles[
            (df_temporal_profiles['category'] == category)
            & (df_temporal_profiles['policy'] == policy)
        ]
        for outcome in ('Residual', '(1,1)', '(2,2)', 'Off diagonal'):
            series = frame[frame['outcome'] == outcome]
            ax.plot(
                series['window_end_round'], np.maximum(series['share'], profile_floor),
                marker='o', linewidth=1.8, color=PROFILE_COLORS[outcome], label=outcome,
            )
        selected = df_selected_cells[
            (df_selected_cells.category == category)
            & (df_selected_cells.policy == policy)
        ].iloc[0]
        ax.set_title(
            f"{POLICY_LABELS[policy]}: {category}\n"
            f"biases=({selected.bias_p1:.3g}, {selected.bias_p2:.3g})",
            fontsize=10,
        )
        ax.set_yscale('log')
        ax.grid(alpha=0.35, which='both', color='#c7dcef')
        if column_index == 0:
            ax.set_ylabel('Window share, logarithmic scale')
        if row_index == len(CATEGORY_ORDER) - 1:
            ax.set_xlabel('Window end round')
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(
    handles, labels, loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.012),
    frameon=False,
)
fig.suptitle('Residual and Off-Equilibrium Play at Selected Cells', fontsize=16, y=1.01)
fig.savefig(DIAGNOSTIC_DIR / 'selected_cell_windowed_residual_profiles.png', dpi=240, bbox_inches='tight')
plt.show()
plt.close(fig)

# Compact paper candidate: substantive initializations only.
paper_categories = ['Near neutral', 'Strongly late-biased', 'Asymmetric']
fig, axes = plt.subplots(3, 3, figsize=(18, 12), constrained_layout=True, sharey=True)
for row_index, category in enumerate(paper_categories):
    for column_index, policy in enumerate(POLICY_ORDER):
        ax = axes[row_index, column_index]
        frame = df_temporal_profiles[
            (df_temporal_profiles['category'] == category)
            & (df_temporal_profiles['policy'] == policy)
        ]
        for outcome in ('Residual', '(1,1)', '(2,2)', 'Off diagonal'):
            series = frame[frame['outcome'] == outcome]
            ax.plot(
                series['window_end_round'], np.maximum(series['share'], profile_floor),
                marker='o', linewidth=2.0, color=PROFILE_COLORS[outcome], label=outcome,
            )
        selected = df_selected_cells[
            (df_selected_cells.category == category)
            & (df_selected_cells.policy == policy)
        ].iloc[0]
        ax.set_title(
            f"{POLICY_LABELS[policy]}: {category}\n"
            f"biases=({selected.bias_p1:.3g}, {selected.bias_p2:.3g})",
            fontsize=10,
        )
        ax.set_yscale('log')
        ax.grid(alpha=0.35, which='both', color='#c7dcef')
        if column_index == 0:
            ax.set_ylabel('Window share, logarithmic scale')
        if row_index == len(paper_categories) - 1:
            ax.set_xlabel('Window end round')
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(
    handles, labels, loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.018),
    frameon=False,
)
fig.suptitle('Residual Play Across Three Initial Conditions', fontsize=16, y=1.015)
fig.savefig(
    DIAGNOSTIC_DIR / 'paper_selected_cell_windowed_residual_profiles.png',
    dpi=300, bbox_inches='tight',
)
fig.savefig(
    PAPER_ASSET_DIR / 'Figure 6.3 selected residual profiles.png',
    dpi=300, bbox_inches='tight',
)
plt.show()
plt.close(fig)



In [ ]:
# =============================================================================
# CELL 15: BINARY TERMINAL CLASSIFICATION AS A CONSISTENCY CHECK
# =============================================================================
classification_check = (
    df_grid.groupby('policy', as_index=False)
    .agg(
        mean_classified_p_stage0=('p_stage0', 'mean'),
        minimum_classified_p_stage0=('p_stage0', 'min'),
        maximum_outcome_entropy=('outcome_entropy', 'max'),
        minimum_horizon=('final_horizon', 'min'),
        maximum_horizon=('final_horizon', 'max'),
    )
)
classification_check['Policy'] = classification_check['policy'].map(POLICY_LABELS)
display(classification_check.drop(columns='policy'))
print(
    'The binary classification is intentionally secondary: it collapses continuous '
    'terminal differences after every replicate clears the classification threshold.'
)

# Paper-ready descriptive tables.
df_paper_policy_summary = df_policy_summary.copy()

def directional_bias_region(row):
    if row.bias_p1 < 0 and row.bias_p2 < 0:
        return 'Both initially favor earlier taking'
    if row.bias_p1 > 0 and row.bias_p2 > 0:
        return 'Both initially favor later taking'
    return 'Opposing initial directions'

df_paper_bias_regions = df_grid.copy()
df_paper_bias_regions['Initial-bias region'] = df_paper_bias_regions.apply(
    directional_bias_region, axis=1
)
df_paper_bias_summary = (
    df_paper_bias_regions
    .groupby(['policy', 'Initial-bias region'], as_index=False)
    .agg(
        Cells=('grid_i', 'size'),
        **{
            'Fraction reaching attraction': ('fraction_reaching_attraction', 'mean'),
            'Median cell first-attraction time': ('median_first_attraction_time', 'median'),
            'Mean terminal (0,0) share': ('mean_terminal_stage0_share', 'mean'),
            'Mean terminal residual share': ('mean_terminal_residual_share', 'mean'),
            'Mean probability of a later exit': ('fraction_exiting_after_entry', 'mean'),
            'Mean re-entries': ('mean_reentries', 'mean'),
        },
    )
)
df_paper_bias_summary.insert(
    0, 'Policy', df_paper_bias_summary['policy'].map(POLICY_LABELS)
)
df_paper_bias_summary = df_paper_bias_summary.drop(columns='policy')
display(df_paper_policy_summary)
display(df_paper_bias_summary)



In [ ]:
# =============================================================================
# CELL 16: PACKAGE COMPLETENESS AND STORAGE INVENTORY
# =============================================================================
inventory_rows = []
for policy in POLICY_ORDER:
    files = sorted((RUN_DIR / 'cells' / policy).glob('cell_*.npz'))
    inventory_rows.append({
        'Policy': POLICY_LABELS[policy],
        'Cell files': len(files),
        'Expected cell files': int(manifest['grid_resolution']) ** 2,
        'Compressed size (MB)': sum(path.stat().st_size for path in files) / 1024 ** 2,
    })
df_inventory = pd.DataFrame(inventory_rows)
display(df_inventory)
if not (df_inventory['Cell files'] == df_inventory['Expected cell files']).all():
    raise ValueError('Run package is missing one or more completed grid-cell files')


In [ ]:
# =============================================================================
# CELL 17: TIMESTAMPED DIAGNOSTIC EXPORTS
# =============================================================================
df_validation.to_csv(DIAGNOSTIC_DIR / 'validation.csv', index=False)
df_policy_summary.to_csv(DIAGNOSTIC_DIR / 'policy_summary.csv', index=False)
df_paper_policy_summary.to_csv(DIAGNOSTIC_DIR / 'paper_policy_summary.csv', index=False)
df_paper_bias_summary.to_csv(DIAGNOSTIC_DIR / 'paper_bias_region_summary.csv', index=False)
df_paper_policy_summary.to_csv(
    PAPER_ASSET_DIR / 'Table 6.1 policy summary.csv', index=False
)
df_paper_bias_summary.to_csv(
    PAPER_ASSET_DIR / 'Supplementary Table S1 bias-region summary.csv', index=False
)
cell_dynamics.to_csv(DIAGNOSTIC_DIR / 'cell_dynamics_summary.csv', index=False)
df_final_state_cells.to_csv(DIAGNOSTIC_DIR / 'final_state_cell_summary.csv', index=False)
df_final_state_summary.to_csv(DIAGNOSTIC_DIR / 'final_state_policy_summary.csv', index=False)
df_value_clock_cells.to_csv(DIAGNOSTIC_DIR / 'mixed_value_clock_cell_summary.csv', index=False)
df_value_clock_summary.to_csv(DIAGNOSTIC_DIR / 'mixed_value_clock_policy_summary.csv', index=False)
df_selected_cells.to_csv(DIAGNOSTIC_DIR / 'selected_temporal_cells.csv', index=False)
df_temporal_profiles.to_csv(DIAGNOSTIC_DIR / 'selected_temporal_profiles.csv', index=False)
classification_check.to_csv(DIAGNOSTIC_DIR / 'terminal_classification_check.csv', index=False)
df_grid.to_csv(DIAGNOSTIC_DIR / 'enriched_grid_summary.csv', index=False)
df_inventory.to_csv(DIAGNOSTIC_DIR / 'storage_inventory.csv', index=False)
print(f'Diagnostic exports written to: {DIAGNOSTIC_DIR.resolve()}')
print(f'Manuscript figures and selected tables written to: {PAPER_ASSET_DIR.resolve()}')


In [ ]:
# =============================================================================
# CELL 18: INTERPRETIVE BOUNDARY
# =============================================================================
print(r"""
These diagnostics describe replicated finite-horizon attraction over explicit
two-dimensional initial-bias slices. Continuous terminal shares, first-entry
times, and exit behavior reveal differences hidden by the common terminal
classification. They do not cover every possible policy state and do not turn
finite simulations into an asymptotic convergence theorem.

The five 100,000-round windows show coarse temporal movement. They cannot
reconstruct the detailed early path or a local vector field because most first
entries occur before the first window closes. The mixed value-clock figures are
final-endpoint diagnostics and are not evidence that PublicationDmix occurs.
""")
